<a href="https://colab.research.google.com/github/priyadarshini-GVPW/XAI_ML/blob/main/Malware%2BBenign.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler


In [2]:
# Load dataset
df = pd.read_csv(
    "https://raw.githubusercontent.com/priyadarshini-GVPW/XAI_ML/main/drebin-215-dataset-5560malware-9476-benign.csv",
    na_values='?',
    low_memory=False
)


print("Dataset shape:", df.shape)
print(df.head())


Dataset shape: (15036, 216)
   transact  onServiceConnected  bindService  attachInterface  \
0         0                   0            0                0   
1         0                   0            0                0   
2         0                   0            0                0   
3         0                   0            0                0   
4         0                   0            0                0   

   ServiceConnection  android.os.Binder  SEND_SMS  \
0                  0                  0         1   
1                  0                  0         1   
2                  0                  0         1   
3                  0                  0         0   
4                  0                  0         0   

   Ljava.lang.Class.getCanonicalName  Ljava.lang.Class.getMethods  \
0                                  0                            0   
1                                  0                            0   
2                                  0                   

In [3]:
df.fillna(0, inplace=True)


In [4]:
print(df.dtypes)


transact                       int64
onServiceConnected             int64
bindService                    int64
attachInterface                int64
ServiceConnection              int64
                               ...  
ACCESS_FINE_LOCATION           int64
SET_WALLPAPER_HINTS            int64
SET_PREFERRED_APPLICATIONS     int64
WRITE_SECURE_SETTINGS          int64
class                         object
Length: 216, dtype: object


In [5]:
# Convert class labels
df['class'] = df['class'].replace({'B': 0, 'S': 1})

print(df['class'].value_counts())


class
0    9476
1    5560
Name: count, dtype: int64


/tmp/ipykernel_6428/191286525.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['class'] = df['class'].replace({'B': 0, 'S': 1})


In [6]:
X = df.drop("class", axis=1)
y = df["class"]

print("Feature shape:", X.shape)
print("Label shape:", y.shape)


Feature shape: (15036, 215)
Label shape: (15036,)


In [7]:
# Scale features for chi2
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Select top 150 features
selector = SelectKBest(chi2, k=150)
X_selected = selector.fit_transform(X_scaled, y)

print("Reduced feature shape:", X_selected.shape)


Reduced feature shape: (15036, 150)


In [8]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

malware_only = X_selected[y == 1]

scaler = StandardScaler()
malware_scaled = scaler.fit_transform(malware_only)

db = DBSCAN(eps=2, min_samples=5)
clusters = db.fit_predict(malware_scaled)

print("Clusters found:", len(set(clusters)))


Clusters found: 188


In [9]:
# ==========================================================
# ANDROID MALWARE DETECTION – COMPLETE PROJECT PIPELINE
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# ==========================================================
# 1️⃣ LOAD AND CLEAN DATASET
# ==========================================================

print("Loading dataset...")

df = pd.read_csv(
    "https://raw.githubusercontent.com/priyadarshini-GVPW/XAI_ML/main/drebin-215-dataset-5560malware-9476-benign.csv",
    na_values='?',
    low_memory=False
)

df.fillna(0, inplace=True)

# Convert class column
df['class'] = df['class'].replace({'B': 0, 'S': 1, 'M': 1, 'SMB': 1})

# Ensure numeric
for col in df.columns:
    if col != "class":
        df[col] = pd.to_numeric(df[col], errors='coerce')

df.fillna(0, inplace=True)

print("Dataset shape:", df.shape)
print(df['class'].value_counts())

# ==========================================================
# 2️⃣ FEATURE SELECTION (CHI-SQUARE)
# ==========================================================

X = df.drop("class", axis=1)
y = df["class"]

selector = SelectKBest(chi2, k=150)
X_selected = selector.fit_transform(X, y)

print("Reduced feature shape:", X_selected.shape)

# ==========================================================
# 3️⃣ TRAIN-TEST SPLIT
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

# ==========================================================
# 4️⃣ RANDOM FOREST MODEL
# ==========================================================

print("\nTraining Random Forest...")

rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)

print("\nRandom Forest Results")
print("Accuracy:", rf_accuracy)
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

# ==========================================================
# 5️⃣ SVM MODEL
# ==========================================================

print("\nTraining SVM...")

svm = LinearSVC(max_iter=5000)
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

svm_accuracy = accuracy_score(y_test, y_pred_svm)

print("\nSVM Results")
print("Accuracy:", svm_accuracy)
print(confusion_matrix(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

# ==========================================================
# 6️⃣ DEEP LEARNING MODEL (MLP)
# ==========================================================

print("\nTraining Deep Learning Model...")

model = Sequential([
    Dense(256, activation='relu', input_shape=(150,)),
    Dropout(0.4),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

loss, dl_accuracy = model.evaluate(X_test, y_test)

print("\nDeep Learning Accuracy:", dl_accuracy)

# ==========================================================
# 7️⃣ MALWARE FAMILY CLUSTERING (DBSCAN)
# ==========================================================

print("\nPerforming Malware Clustering...")

malware_only = X_selected[y == 1]

scaler = StandardScaler()
malware_scaled = scaler.fit_transform(malware_only)

db = DBSCAN(eps=2, min_samples=5)
clusters = db.fit_predict(malware_scaled)

unique_clusters = np.unique(clusters)
num_clusters = len(unique_clusters[unique_clusters != -1])
noise_points = np.sum(clusters == -1)

print("Clusters found (excluding noise):", num_clusters)
print("Noise samples:", noise_points)

if num_clusters > 1:
    sil_score = silhouette_score(malware_scaled, clusters)
    print("Silhouette Score:", sil_score)

# ==========================================================
# 8️⃣ FINAL RESULTS SUMMARY
# ==========================================================

print("\n==================================================")
print("FINAL RESULTS SUMMARY")
print("Random Forest Accuracy:", rf_accuracy)
print("SVM Accuracy:", svm_accuracy)
print("Deep Learning Accuracy:", dl_accuracy)
print("Number of Malware Clusters:", num_clusters)
print("Noise Samples:", noise_points)
print("==================================================")


Loading dataset...


/tmp/ipykernel_6428/2881653424.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['class'] = df['class'].replace({'B': 0, 'S': 1, 'M': 1, 'SMB': 1})


Dataset shape: (15036, 216)
class
0    9476
1    5560
Name: count, dtype: int64
Reduced feature shape: (15036, 150)
Train size: (12028, 150)
Test size: (3008, 150)

Training Random Forest...

Random Forest Results
Accuracy: 0.9863696808510638
[[1881   15]
 [  26 1086]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1896
           1       0.99      0.98      0.98      1112

    accuracy                           0.99      3008
   macro avg       0.99      0.98      0.99      3008
weighted avg       0.99      0.99      0.99      3008


Training SVM...

SVM Results
Accuracy: 0.9763962765957447
[[1870   26]
 [  45 1067]]
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      1896
           1       0.98      0.96      0.97      1112

    accuracy                           0.98      3008
   macro avg       0.98      0.97      0.97      3008
weighted avg       0.98      0.98      0.98  

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


301/301 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9004 - loss: 0.2447 - val_accuracy: 0.9759 - val_loss: 0.0705
Epoch 2/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9732 - loss: 0.0817 - val_accuracy: 0.9805 - val_loss: 0.0532
Epoch 3/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9784 - loss: 0.0639 - val_accuracy: 0.9863 - val_loss: 0.0472
Epoch 4/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9794 - loss: 0.0559 - val_accuracy: 0.9863 - val_loss: 0.0426
Epoch 5/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9852 - loss: 0.0452 - val_accuracy: 0.9879 - val_loss: 0.0403
Epoch 6/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9884 - loss: 0.0399 - val_accuracy: 0.9879 - val_loss: 0.0385
Epoch 7/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9915 - loss: 0.0268 - val_accuracy: 0.9888 - val_loss: 0.0371
Epoch 8/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9897 - loss: 0.0310 - val_accuracy: 0.9892 - val_

In [10]:
# ==========================================================
# CNN + LSTM MODEL
# ==========================================================

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM
from tensorflow.keras.layers import Dense, Dropout, Input, Flatten
from tensorflow.keras.optimizers import Adam

print("\nPreparing data for CNN-LSTM...")

# Reshape data to 3D
X_train_seq = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_seq = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("New training shape:", X_train_seq.shape)

# ==========================================================
# BUILD CNN + LSTM MODEL
# ==========================================================

model_cnn_lstm = Sequential([
    Input(shape=(150, 1)),

    # CNN Layer
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),

    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),

    # LSTM Layer
    LSTM(64, return_sequences=False),

    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

model_cnn_lstm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_cnn_lstm.summary()

# ==========================================================
# TRAIN MODEL
# ==========================================================

history = model_cnn_lstm.fit(
    X_train_seq,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# ==========================================================
# EVALUATE
# ==========================================================

loss, cnn_lstm_accuracy = model_cnn_lstm.evaluate(X_test_seq, y_test)

print("\nCNN-LSTM Test Accuracy:", cnn_lstm_accuracy)




Preparing data for CNN-LSTM...
New training shape: (12028, 150, 1)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 148, 64)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 74, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 72, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 36, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 78,593 (307.00 KB)

 Trainable params: 78,593 (307.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 17s 47ms/step - accuracy: 0.6253 - loss: 0.6452 - val_accuracy: 0.7311 - val_loss: 0.5000
Epoch 2/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 14s 45ms/step - accuracy: 0.6815 - loss: 0.5833 - val_accuracy: 0.6322 - val_loss: 0.6564
Epoch 3/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - accuracy: 0.6439 - loss: 0.6445 - val_accuracy: 0.8080 - val_loss: 0.4202
Epoch 4/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - accuracy: 0.7079 - loss: 0.5538 - val_accuracy: 0.6322 - val_loss: 0.6556
Epoch 5/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 21s 48ms/step - accuracy: 0.6325 - loss: 0.6589 - val_accuracy: 0.6322 - val_loss: 0.6551
Epoch 6/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 20s 45ms/step - accuracy: 0.6305 - loss: 0.6595 - val_accuracy: 0.6322 - val_loss: 0.6531
Epoch 7/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 14s 46ms/step - accuracy: 0.6199 - loss: 0.6635 - val_accuracy: 0.6322 - val_loss: 0.6521
Epoch 8/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 14s 46ms/step - accuracy: 0.6328 - loss: 0.6543 - 

In [11]:
!pip install androguard shap gradio --quiet


In [12]:
import pandas as pd
import pickle
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2

# Load dataset
df = pd.read_csv(
    "https://raw.githubusercontent.com/priyadarshini-GVPW/XAI_ML/main/drebin-215-dataset-5560malware-9476-benign.csv",
    na_values='?',
    low_memory=False
)

df.fillna(0, inplace=True)
df['class'] = df['class'].replace({'B':0,'S':1,'M':1,'SMB':1})

for col in df.columns:
    if col != "class":
        df[col] = pd.to_numeric(df[col], errors='coerce')

df.fillna(0, inplace=True)

X = df.drop("class", axis=1)
y = df["class"]

feature_columns = X.columns.tolist()

# Feature Selection
selector = SelectKBest(chi2, k=150)
X_selected = selector.fit_transform(X, y)

# Train RF
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_selected, y)

# Save models
os.makedirs("saved_models", exist_ok=True)

pickle.dump(rf, open("saved_models/rf_model.pkl", "wb"))
pickle.dump(selector, open("saved_models/selector.pkl", "wb"))
pickle.dump(feature_columns, open("saved_models/feature_columns.pkl", "wb"))

print("Model trained and saved successfully.")


/tmp/ipykernel_6428/2834747453.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['class'] = df['class'].replace({'B':0,'S':1,'M':1,'SMB':1})


Model trained and saved successfully.


In [13]:
from androguard.core.apk import APK

def extract_features(apk_path):

    apk = APK(apk_path)

    permissions = apk.get_permissions()
    activities = apk.get_activities()
    services = apk.get_services()
    receivers = apk.get_receivers()

    feature_columns = pickle.load(open("saved_models/feature_columns.pkl", "rb"))

    feature_vector = {col: 0 for col in feature_columns}

    for perm in permissions:
        if perm in feature_vector:
            feature_vector[perm] = 1

    for act in activities:
        if act in feature_vector:
            feature_vector[act] = 1

    for srv in services:
        if srv in feature_vector:
            feature_vector[srv] = 1

    for rec in receivers:
        if rec in feature_vector:
            feature_vector[rec] = 1

    df = pd.DataFrame([feature_vector])

    return df


In [14]:
import shap
import numpy as np

rf = pickle.load(open("saved_models/rf_model.pkl", "rb"))
selector = pickle.load(open("saved_models/selector.pkl", "rb"))

def predict_apk(apk_path):

    df = extract_features(apk_path)
    X_selected = selector.transform(df)

    prediction = rf.predict(X_selected)[0]
    probability = rf.predict_proba(X_selected)[0][1]

    # SHAP explanation
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_selected)

    top_indices = np.argsort(np.abs(shap_values[1][0]))[-10:]
    selected_features = selector.get_feature_names_out()

    important_features = [selected_features[i] for i in top_indices]

    label = "Malware" if prediction == 1 else "Benign"

    return label, probability, important_features
